# 🤖 Tahap 3–4: Encoding TF-IDF & Modeling

Notebook ini menjalankan:
- **Encoding label** (LabelEncoder)
- **TF-IDF Vectorization** (unigram + bigram, max 10.000 fitur)
- **Train-test split** stratified (80:20)
- **SMOTE** untuk menangani data tidak seimbang
- **Training 6 model**: LR, SVM, NB × (Sebelum SMOTE, Sesudah SMOTE)

**Input:** `data/processed/dataset_clean_manual.csv`

**Output (disimpan ke `output/`):**
- Model `.pkl` (6 file)
- `tfidf_vectorizer.pkl`
- `label_encoder.pkl`
- `X_test.npz` & `y_test.npy` (untuk notebook 04 & 05)
- `model_names.json`

In [1]:
import os
import sys

def _find_root(start):
    """Cari root directory project (berisi folder 'src' dan 'data')."""
    d = start
    for _ in range(5):
        if os.path.isdir(os.path.join(d, 'src')) and os.path.isdir(os.path.join(d, 'data')):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            break
        d = parent
    return start

ROOT_DIR   = _find_root(os.path.abspath(os.getcwd()))
SRC_DIR    = os.path.join(ROOT_DIR, 'src')
OUTPUT_DIR = os.path.join(ROOT_DIR, 'output')

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'ROOT_DIR   : {ROOT_DIR}')
print(f'SRC_DIR    : {SRC_DIR}')
print(f'OUTPUT_DIR : {OUTPUT_DIR}')

ROOT_DIR   : D:\Kuliah\Semester 6\Data Mining\Tubes\tugas-besar-datamining-kelompok3
SRC_DIR    : D:\Kuliah\Semester 6\Data Mining\Tubes\tugas-besar-datamining-kelompok3\src
OUTPUT_DIR : D:\Kuliah\Semester 6\Data Mining\Tubes\tugas-besar-datamining-kelompok3\output


In [2]:
import json
import numpy as np
import scipy.sparse

from data_loader import load_processed_data
from model import (
    encode_labels,
    tfidf_vectorize,
    split_data_stratified,
    apply_smote,
    train_all_models,
    save_models
)

In [3]:
df_modeling = load_processed_data('dataset_clean_manual.csv')

print(f'Jumlah data : {len(df_modeling):,} baris')
print(f'Kolom       : {df_modeling.columns.tolist()}')
df_modeling.head()

[data_loader] Processed dataset 'dataset_clean_manual.csv' dimuat: 10295 baris, 6 kolom.
Jumlah data : 10,295 baris
Kolom       : ['id', 'text', 'text_clean', 'label_inset_lexicon', 'label_manual', 'catatan']


,id,text,text_clean,label_inset_lexicon,label_manual,catatan
0,10101,mending gua aja dah yg jadi presiden,mending gua saja presiden,Negatif,Negatif,NaN
1,3001,ternyata sama wakilnya,nyata sama wakil,Netral,Netral,NaN
2,6886,hore dolar naik lagi indonesia hebat,hore dolar naik indonesia hebat,Netral,Negatif,NaN
3,10007,untung dulu gua gak milih dia,untung dulu gua tidak pilih,Negatif,Negatif,NaN
4,3300,ini contoh manusia sakitbrjelas kan,contoh manusia sakitbrjelas,Positif,Negatif,NaN


In [4]:
def standarkan_label(label):
    if str(label).strip() == '' or str(label).lower() == 'nan':
        return None
    label = str(label).strip().lower()
    mapping = {
        'positif': 'Positif', 'pos': 'Positif', 'p': 'Positif',
        'negatif': 'Negatif', 'neg': 'Negatif', 'n': 'Negatif',
        'netral':  'Netral',  'net': 'Netral',
    }
    return mapping.get(label, label)

df_modeling['label_manual'] = df_modeling['label_manual'].apply(standarkan_label)
df_modeling = df_modeling[
    df_modeling['label_manual'].isin(['Positif', 'Negatif', 'Netral'])
].reset_index(drop=True)

print('\nDistribusi label final:')
print(df_modeling['label_manual'].value_counts())


Distribusi label final:
label_manual
Negatif    8342
Netral     1486
Positif     461
Name: count, dtype: int64


In [5]:
df_modeling, label_encoder = encode_labels(df_modeling, label_col='label_manual')

X_text = df_modeling['text_clean']
y      = df_modeling['label_encoded']

print(f'\nJumlah data fitur (X) : {len(X_text):,}')
print(f'Jumlah data target (y): {len(y):,}')

# TF-IDF
X_tfidf, tfidf = tfidf_vectorize(X_text)

[model] Mapping Label Encoding:
  Negatif    → 0
  Netral     → 1
  Positif    → 2

Jumlah data fitur (X) : 10,289
Jumlah data target (y): 10,289


[model] Bentuk matriks TF-IDF : (10289, 10000)
  → 10,289 baris (dokumen)
  → 10,000 kolom (fitur kata/frasa unik)

[model] Contoh 10 fitur pertama:
['abad' 'abah' 'abai' 'abal' 'abal abal' 'abal pesan' 'abang'
 'abang presiden' 'abdi' 'abis']


In [6]:
X_train, X_test, y_train, y_test = split_data_stratified(X_tfidf, y)

print(f'\nDistribusi y_train:')
import pandas as pd
print(pd.Series(y_train).value_counts())

# Terapkan SMOTE
X_train_smote, y_train_smote = apply_smote(X_train, y_train)

[model] Data latih (train) : 8,231 baris
[model] Data uji (test)    : 2,058 baris

Distribusi y_train:
label_encoded
0    6673
1    1189
2     369
Name: count, dtype: int64
[model] Distribusi setelah SMOTE:
  Kelas 0: 6,673 sampel
  Kelas 1: 6,673 sampel
  Kelas 2: 6,673 sampel


In [7]:
models = train_all_models(X_train, y_train, X_train_smote, y_train_smote)


[model] ===== MODELING SEBELUM SMOTE =====


[model] Logistic Regression selesai dilatih.


[model] SVM selesai dilatih.
[model] Naive Bayes selesai dilatih.

[model] ===== MODELING SESUDAH SMOTE =====


[model] Logistic Regression selesai dilatih.


[model] SVM selesai dilatih.
[model] Naive Bayes selesai dilatih.


In [8]:
save_models(models, tfidf, label_encoder, save_dir=OUTPUT_DIR)

[model] Model disimpan: D:\Kuliah\Semester 6\Data Mining\Tubes\tugas-besar-datamining-kelompok3\output\LR_Sebelum_SMOTE.pkl
[model] Model disimpan: D:\Kuliah\Semester 6\Data Mining\Tubes\tugas-besar-datamining-kelompok3\output\SVM_Sebelum_SMOTE.pkl
[model] Model disimpan: D:\Kuliah\Semester 6\Data Mining\Tubes\tugas-besar-datamining-kelompok3\output\NB_Sebelum_SMOTE.pkl
[model] Model disimpan: D:\Kuliah\Semester 6\Data Mining\Tubes\tugas-besar-datamining-kelompok3\output\LR_Sesudah_SMOTE.pkl
[model] Model disimpan: D:\Kuliah\Semester 6\Data Mining\Tubes\tugas-besar-datamining-kelompok3\output\SVM_Sesudah_SMOTE.pkl
[model] Model disimpan: D:\Kuliah\Semester 6\Data Mining\Tubes\tugas-besar-datamining-kelompok3\output\NB_Sesudah_SMOTE.pkl
[model] TF-IDF vectorizer disimpan: D:\Kuliah\Semester 6\Data Mining\Tubes\tugas-besar-datamining-kelompok3\output\tfidf_vectorizer.pkl
[model] Label encoder disimpan: D:\Kuliah\Semester 6\Data Mining\Tubes\tugas-besar-datamining-kelompok3\output\label_e

In [9]:
scipy.sparse.save_npz(os.path.join(OUTPUT_DIR, 'X_test.npz'), X_test)

np.save(os.path.join(OUTPUT_DIR, 'y_test.npy'), y_test.values)

with open(os.path.join(OUTPUT_DIR, 'model_names.json'), 'w') as f:
    json.dump(list(models.keys()), f, ensure_ascii=False, indent=2)

print('[modeling] X_test.npz  disimpan ke output/')
print('[modeling] y_test.npy  disimpan ke output/')
print('[modeling] model_names.json disimpan ke output/')
print(f'\nModel yang disimpan: {list(models.keys())}')

[modeling] X_test.npz  disimpan ke output/
[modeling] y_test.npy  disimpan ke output/
[modeling] model_names.json disimpan ke output/

Model yang disimpan: ['LR (Sebelum SMOTE)', 'SVM (Sebelum SMOTE)', 'NB (Sebelum SMOTE)', 'LR (Sesudah SMOTE)', 'SVM (Sesudah SMOTE)', 'NB (Sesudah SMOTE)']
